# 比例控制图完整教程：p 图与 np 图

## 📚 教学目标
1. 理解比例控制图的适用场景：监控不合格品率
2. 掌握 p 控制图（不合格品率控制图）的构建步骤
3. 掌握 np 控制图（不合格品数控制图）的构建步骤
4. 理解样本量变化对控制限的影响
5. 用 Python 实现完整的比例控制图分析

**参考书**：《基础统计学(第14版)》（Triola）第 14-2 节
**教学策略**：先用手算理解 p 图的控制限公式，再扩展到生产线监控

---

## 1. 场景设定：为什么需要比例控制图？

### 🎯 一个现实问题

一家电子元件工厂每天检测 100 个产品，记录不合格品数量。

**核心问题**：不合格品率的波动是正常的随机变异，还是生产线出了问题？

### 📖 书中的定义

> p 控制图用于监控制过程的不合格品率。
> 当不合格品率超出控制限时，说明过程可能失控，需要排查原因。

### 💡 p 图 vs np 图

| 控制图 | 监控指标 | 样本量要求 |
|--------|----------|------------|
| p 图 | 不合格品率 p̂ | 样本量可以变化 |
| np 图 | 不合格品数 np̂ | 样本量必须固定 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据手算：p 控制图

### 📊 数据

假设我们有 10 天的检测数据：

| 天 | 检测数 n | 不合格品数 | 不合格品率 p̂ |
|----|---------|-----------|--------------|
| 1 | 100 | 5 | 0.05 |
| 2 | 100 | 7 | 0.07 |
| 3 | 100 | 4 | 0.04 |
| 4 | 100 | 6 | 0.06 |
| 5 | 100 | 8 | 0.08 |
| 6 | 100 | 3 | 0.03 |
| 7 | 100 | 6 | 0.06 |
| 8 | 100 | 5 | 0.05 |
| 9 | 100 | 7 | 0.07 |
| 10 | 100 | 4 | 0.04 |

### 📐 p 控制图的控制限公式

$$UCL = \bar{p} + 3\sqrt{\frac{\bar{p}(1-\bar{p})}{n}}$$
$$CL = \bar{p}$$
$$LCL = \bar{p} - 3\sqrt{\frac{\bar{p}(1-\bar{p})}{n}}$$

其中：
- $\bar{p}$ = 平均不合格品率
- $n$ = 样本量
- $\bar{p}(1-\bar{p})$ = 二项分布的方差

In [ ]:
# ========== 步骤 1: 微型数据集 ==========

days = np.arange(1, 11)
n_per_day = np.full(10, 100)  # 每天检测 100 个
defectives = np.array([5, 7, 4, 6, 8, 3, 6, 5, 7, 4])
p_hat = defectives / n_per_day

print("📊 步骤 1: 微型数据集")
print(f"  {'天':<4} {'检测数 n':<10} {'不合格品数':<12} {'不合格品率 p̂':<15}")
print(f"  {'-'*41}")
for i in range(10):
    print(f"  {days[i]:<4} {n_per_day[i]:<10} {defectives[i]:<12} {p_hat[i]:<15.2f}")

In [ ]:
# ========== 步骤 2: 计算平均不合格品率 ==========

total_defectives = np.sum(defectives)
total_inspected = np.sum(n_per_day)
p_bar = total_defectives / total_inspected

print("📊 步骤 2: 计算平均不合格品率")
print(f"  总不合格品数 = {total_defectives}")
print(f"  总检测数 = {total_inspected}")
print(f"  p̄ = {total_defectives}/{total_inspected} = {p_bar:.4f}")

In [ ]:
# ========== 步骤 3: 计算控制限 ==========

n_fixed = n_per_day[0]  # 固定样本量
sigma_p = np.sqrt(p_bar * (1 - p_bar) / n_fixed)

UCL = p_bar + 3 * sigma_p
CL = p_bar
LCL = max(0, p_bar - 3 * sigma_p)  # LCL 不能为负

print("📊 步骤 3: 计算 p 控制图的控制限")
print(f"  样本量 n = {n_fixed}")
print(f"  标准误 σ_p = √(p̄(1-p̄)/n) = √({p_bar:.4f}×{1-p_bar:.4f}/{n_fixed}) = {sigma_p:.4f}")
print(f"  UCL = p̄ + 3σ_p = {p_bar:.4f} + 3×{sigma_p:.4f} = {UCL:.4f}")
print(f"  CL  = p̄ = {CL:.4f}")
print(f"  LCL = p̄ - 3σ_p = {p_bar:.4f} - 3×{sigma_p:.4f} = {LCL:.4f}")

In [ ]:
# ========== 步骤 4: 可视化 p 控制图 ==========

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(days, p_hat, 'o-', color='steelblue', linewidth=2, markersize=8, label='Sample Proportion p̂')
ax.axhline(y=UCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL:.4f}')
ax.axhline(y=CL, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL:.4f}')
ax.axhline(y=LCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL:.4f}')
ax.fill_between(days, UCL, LCL, alpha=0.05, color='steelblue')

# 标记失控点
ooc = (p_hat > UCL) | (p_hat < LCL)
if np.any(ooc):
    ax.scatter(days[ooc], p_hat[ooc], color='#e74c3c', s=100, zorder=5, label='Out of Control')

ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Defective Rate', fontsize=12)
ax.set_title('p Control Chart (Defective Rate)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xticks(days)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  蓝色线：每天的不合格品率")
print("  绿色实线：中心线（平均不合格品率）")
print("  红色虚线：控制限（±3σ）")
print("  所有点都在控制限内 → 过程处于统计控制状态")

---

## 3. np 控制图（不合格品数控制图）

### 📐 np 控制图的控制限公式

当样本量固定时，可以直接监控不合格品数：

$$UCL = n\bar{p} + 3\sqrt{n\bar{p}(1-\bar{p})}$$
$$CL = n\bar{p}$$
$$LCL = n\bar{p} - 3\sqrt{n\bar{p}(1-\bar{p})}$$

### 💡 p 图 vs np 图

p 图和 np 图本质上监控同一个过程，只是纵坐标不同：
- p 图：纵坐标是比例（不合格品率）
- np 图：纵坐标是数量（不合格品数）

In [ ]:
# ========== 步骤 5: np 控制图 ==========

np_bar = n_fixed * p_bar
sigma_np = np.sqrt(n_fixed * p_bar * (1 - p_bar))

UCL_np = np_bar + 3 * sigma_np
CL_np = np_bar
LCL_np = max(0, np_bar - 3 * sigma_np)

print("📊 步骤 5: 计算 np 控制图的控制限")
print(f"  n = {n_fixed}")
print(f"  n*p̄ = {np_bar:.4f}")
print(f"  σ_np = √(n*p̄*(1-p̄)) = √({n_fixed}×{p_bar:.4f}×{1-p_bar:.4f}) = {sigma_np:.4f}")
print(f"  UCL = n*p̄ + 3σ_np = {UCL_np:.4f}")
print(f"  CL  = n*p̄ = {CL_np:.4f}")
print(f"  LCL = n*p̄ - 3σ_np = {LCL_np:.4f}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# p 图
ax1 = axes[0]
ax1.plot(days, p_hat, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.axhline(y=UCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL:.4f}')
ax1.axhline(y=CL, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL:.4f}')
ax1.axhline(y=LCL, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL:.4f}')
ax1.set_xlabel('Day', fontsize=12)
ax1.set_ylabel('Defective Rate', fontsize=12)
ax1.set_title('p Chart', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_xticks(days)

# np 图
ax2 = axes[1]
ax2.plot(days, defectives, 'o-', color='steelblue', linewidth=2, markersize=8)
ax2.axhline(y=UCL_np, color='#e74c3c', linestyle='--', linewidth=2, label=f'UCL = {UCL_np:.4f}')
ax2.axhline(y=CL_np, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {CL_np:.4f}')
ax2.axhline(y=LCL_np, color='#e74c3c', linestyle='--', linewidth=2, label=f'LCL = {LCL_np:.4f}')
ax2.set_xlabel('Day', fontsize=12)
ax2.set_ylabel('Number of Defectives', fontsize=12)
ax2.set_title('np Chart', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_xticks(days)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：p 图 — 监控不合格品率")
print("  右图：np 图 — 监控不合格品数")
print("  两张图提供相同的信息，只是纵坐标不同")

---

## 4. 大样本扩展：样本量变化的情况

### 📐 样本量变化的影响

在实际生产中，每天的检测数量可能不同。当样本量变化时：
- p 图的控制限不再是水平直线
- np 图不适用（因为 n 不固定）

### 📐 数据生成过程 (DGP)

模拟 30 天的检测数据：
- 每天检测数量：n ∈ [80, 120]（随机变化）
- 正常不合格品率：p = 0.05
- 在第 21 天后，不合格品率上升到 p = 0.10

In [ ]:
# ========== 大样本模拟：样本量变化 ==========

# --- 数据生成过程 (DGP) ---
k_days = 30
n_variable = np.random.randint(80, 121, k_days)  # 每天检测 80-120 个

# 前 20 天：正常过程 p = 0.05
defectives_normal = np.random.binomial(n_variable[:20], 0.05)
# 后 10 天：不合格品率上升 p = 0.10
defectives_shift = np.random.binomial(n_variable[20:], 0.10)
defectives_all = np.concatenate([defectives_normal, defectives_shift])
p_hat_all = defectives_all / n_variable

print(f"📊 大样本模拟参数:")
print(f"  天数 k = {k_days}")
print(f"  样本量范围: [{n_variable.min()}, {n_variable.max()}]")
print(f"  正常不合格品率: p = 0.05")
print(f"  异常不合格品率: p = 0.10 (第 21 天后)")

In [ ]:
# ========== 计算控制限（用前 20 天数据） ==========

# 用前 20 天正常数据计算 p̄
p_bar_est = np.sum(defectives_normal) / np.sum(n_variable[:20])

print(f"📊 用前 20 天正常数据估计:")
print(f"  p̄ = {p_bar_est:.4f}")

# 每天的控制限（因为 n 不同）
UCL_daily = p_bar_est + 3 * np.sqrt(p_bar_est * (1 - p_bar_est) / n_variable)
CL_daily = np.full(k_days, p_bar_est)
LCL_daily = np.maximum(0, p_bar_est - 3 * np.sqrt(p_bar_est * (1 - p_bar_est) / n_variable))

# 标记失控点
ooc_all = (p_hat_all > UCL_daily) | (p_hat_all < LCL_daily)

print(f"\n📊 各天统计量:")
print(f"  {'天':<4} {'n':<6} {'不合格品数':<10} {'p̂':<8} {'UCL':<8} {'状态':<10}")
print(f"  {'-'*46}")
for i in range(k_days):
    status = '⚠️ 失控' if ooc_all[i] else '✓ 正常'
    marker = ' ← 异常始' if i == 20 else ''
    print(f"  {i+1:<4} {n_variable[i]:<6} {defectives_all[i]:<10} {p_hat_all[i]:<8.3f} {UCL_daily[i]:<8.3f} {status:<10}{marker}")

In [ ]:
# ========== 可视化：样本量变化的 p 控制图 ==========

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

days_idx = np.arange(1, k_days + 1)

# --- p 控制图 ---
ax1 = axes[0]
ax1.plot(days_idx, p_hat_all, 'o-', color='steelblue', linewidth=1.5, markersize=6, label='p̂')
ax1.plot(days_idx, UCL_daily, '--', color='#e74c3c', linewidth=1.5, label='UCL (variable n)')
ax1.axhline(y=p_bar_est, color='#2ecc71', linestyle='-', linewidth=2, label=f'CL = {p_bar_est:.4f}')
ax1.plot(days_idx, LCL_daily, '--', color='#e74c3c', linewidth=1.5, label='LCL (variable n)')

# 标记失控点
if np.any(ooc_all):
    ax1.scatter(days_idx[ooc_all], p_hat_all[ooc_all], color='#e74c3c', s=100, zorder=5, label='Out of Control')

ax1.axvline(x=21, color='#e67e22', linestyle=':', linewidth=2, alpha=0.7, label='Shift Start (Day 21)')
ax1.set_xlabel('Day', fontsize=12)
ax1.set_ylabel('Defective Rate', fontsize=12)
ax1.set_title('p Control Chart with Variable Sample Size', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(alpha=0.3)
ax1.set_xticks(days_idx)

# --- 样本量变化图 ---
ax2 = axes[1]
ax2.bar(days_idx, n_variable, color='steelblue', alpha=0.7, edgecolor='white')
ax2.axhline(y=np.mean(n_variable), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean n = {np.mean(n_variable):.0f}')
ax2.axvline(x=21, color='#e67e22', linestyle=':', linewidth=2, alpha=0.7, label='Shift Start')
ax2.set_xlabel('Day', fontsize=12)
ax2.set_ylabel('Sample Size n', fontsize=12)
ax2.set_title('Daily Sample Size', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, axis='y')
ax2.set_xticks(days_idx)

plt.tight_layout()
plt.show()

n失控 = np.sum(ooc_all)
print(f"\n💡 图解说明：")
print(f"  上图：p 控制图 — 控制限随样本量变化而变化")
print(f"  下图：每天的样本量（柱状图）")
print(f"  橙色虚线：第 21 天开始不合格品率上升")
print(f"  红色圆点：超出控制限的失控点")
print(f"  失控点数: {n失控}/{k_days}")
print(f"  样本量越大，控制限越窄（因为标准误 = √(p(1-p)/n) 越小）")

---

## 5. 控制图的诊断能力

### 📐 模拟：不同偏移量下的检测概率

控制图能否及时发现过程偏移？我们模拟不同偏移量下的检测能力。

In [ ]:
# ========== 模拟：不同偏移量下的检测概率 ==========

p0 = 0.05  # 正常不合格品率
n_sim = 1000  # 模拟次数
n_sample_sim = 100  # 每次检测数量

# 控制限（基于正常过程）
sigma_sim = np.sqrt(p0 * (1 - p0) / n_sample_sim)
UCL_sim = p0 + 3 * sigma_sim
LCL_sim = max(0, p0 - 3 * sigma_sim)

# 不同的偏移量
shifts = [0.00, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]

print("📊 不同偏移量下的失控检测概率:")
print(f"  正常 p = {p0}")
print(f"  样本量 n = {n_sample_sim}")
print(f"  控制限: [{LCL_sim:.4f}, {UCL_sim:.4f}]")
print(f"\n  {'偏移量':<10} {'新 p':<10} {'检测概率':<12}")
print(f"  {'-'*32}")

for shift in shifts:
    p_new = p0 + shift
    # 模拟 n_sim 次，每次抽 n_sample_sim 个
    detections = 0
    for _ in range(n_sim):
        defectives_sim = np.random.binomial(n_sample_sim, p_new)
        p_hat_sim = defectives_sim / n_sample_sim
        if p_hat_sim > UCL_sim or p_hat_sim < LCL_sim:
            detections += 1
    detection_rate = detections / n_sim
    print(f"  {shift:<10.2f} {p_new:<10.2f} {detection_rate:<12.2%}")

print(f"\n💡 解读:")
print(f"  偏移量越大，检测概率越高")
print(f"  小偏移（如 0.01）很难被单个样本检测到")
print(f"  这就是为什么需要累积多个样本的趋势分析")

---

## 📌 核心概念回顾

### 📌 p 控制图 (p Chart)
- **定义**: 监控不合格品率的控制图
- **公式**: UCL = p̄ + 3√(p̄(1-p̄)/n), CL = p̄, LCL = p̄ - 3√(p̄(1-p̄)/n)
- **适用**: 样本量可以变化
- **判断标准**: 超出控制限 → 过程失控

### 📌 np 控制图 (np Chart)
- **定义**: 监控不合格品数的控制图
- **公式**: UCL = np̄ + 3√(np̄(1-p̄)), CL = np̄, LCL = np̄ - 3√(np̄(1-p̄))
- **适用**: 样本量固定
- **判断标准**: 同 p 图

### 📌 样本量的影响
- 样本量越大 → 控制限越窄 → 检测灵敏度越高
- 样本量越小 → 控制限越宽 → 可能漏检小偏移
- 当 n 变化时，p 图的控制限不是水平线

### 📌 检测能力
- 大偏移（>5%）→ 单个样本即可检测
- 小偏移（<2%）→ 需要多个样本的趋势分析
- Western Electric 规则可以提高小偏移的检测能力

### 🔗 完整流程
```
收集数据 → 计算 p̄ → 确定控制限 → 绘制控制图 → 检查失控信号
    ↓          ↓          ↓             ↓             ↓
  k天数据    总不合格/总检测  3σ公式    p̂ vs UCL/LCL  超限/趋势
```

---

## ❌ 常见误区

### ❌ 误区 1: p̄ = 0 就不需要控制图
**✓ 正确理解**: 如果 p̄ = 0（没有不合格品），说明过程可能已经非常好，但也可能是检测方法有问题。需要确认检测系统是否正常工作。

### ❌ 误区 2: 控制限超出 [0, 1] 范围就要截断
**✓ 正确理解**: 当 p̄ 接近 0 或 1 时，正态近似不再适用。此时应使用二项分布的精确控制限，或改用 c 图/u 图。

### ❌ 误区 3: 样本量变化不影响控制图
**✓ 正确理解**: 样本量变化直接影响控制限的宽度。p 图的控制限随 n 变化而变化，必须对每个样本单独计算控制限。

### ❌ 误区 4: 只关注不合格品率，不关注不合格品数
**✓ 正确理解**: 当样本量很大时，即使不合格品率很小，不合格品数也可能很多。应同时关注两者，确保对过程有全面了解。

### ❌ 误区 5: 控制图可以替代规格检验
**✓ 正确理解**: 控制图监控过程稳定性，规格检验监控产品合格性。一个稳定的过程可能产出不合格产品（如果过程中心偏离目标）。两者需要同时使用。